In [128]:
import torch
import torch.nn as nn

from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, GPT2Model

"""
This implementation intentionally does not use GPT2ForSequenceClassification,
because I think manually building the head is more valuable for our current learning goal.
"""

'\nThis implementation intentionally does not use GPT2ForSequenceClassification, \nbecause I think manually building the head is more valuable for our current learning goal.\n'

In [129]:
# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "openai-community/gpt2"

TRAIN_SIZE = 10_000
VAL_SIZE = 872

MAX_LENGTH = 128
BATCH_SIZE = 4
NUM_EPOCHS = 1

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01

SEED = 42

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")


print("Device:", DEVICE)



Device: cuda


In [130]:
# ============================================================
# Reproducibility
# ============================================================

torch.manual_seed(SEED)



In [131]:
# ============================================================
# Load SST-2
# ============================================================

dataset = load_dataset("stanfordnlp/sst2")

print(dataset)

# We use:
#
#   train      -> 10,000 examples
#   validation -> official SST-2 validation set
#
# The official test labels are hidden, so we don't use test here.

train_dataset = (
    dataset["train"]
    .shuffle(seed=SEED)
    .select(range(TRAIN_SIZE))
)

val_dataset = dataset["validation"]


print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))




DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})
Train size: 10000
Validation size: 872


In [135]:
print(dataset["train"][0])
print(dataset["train"][0]["idx"])
print(dataset["train"][0]["sentence"])
print(dataset["train"][0]['label'])

{'idx': 0, 'sentence': 'hide new secretions from the parental units ', 'label': 0}
0
hide new secretions from the parental units 
0


In [132]:
# ============================================================
# Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 does not have a native padding token.
# For batching, use EOS as PAD.
tokenizer.pad_token = tokenizer.eos_token



In [133]:
# ============================================================
# Dataset preprocessing
# ============================================================

def tokenize_example(example):
    encoded = tokenizer(
        example["sentence"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

    encoded["label"] = example["label"]

    return encoded


train_dataset = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)


val_dataset = val_dataset.map(
    tokenize_example,
    remove_columns=val_dataset.column_names,
)


In [134]:
val_dataset

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 872
})

In [136]:
# ============================================================
# Dynamic padding collator
# ============================================================

def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]

    encoded = tokenizer.pad(
        {
            "input_ids": input_ids,
        },
        padding=True,
        return_tensors="pt",
    )

    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long,
    )

    encoded["labels"] = labels

    return encoded




In [137]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)



In [138]:
# ============================================================
# Model
# ============================================================

class GPT2SentimentClassifier(nn.Module):

    def __init__(self, model_name):
        super().__init__()

        # GPT-2 WITHOUT the language-model head.
        self.gpt2 = GPT2Model.from_pretrained(model_name)

        hidden_size = self.gpt2.config.hidden_size

        # Single logits Binary classification head:
        #
        # [B, 768] -> [B, 1]
        self.classifier = nn.Linear(
            hidden_size,
            1,
        )

    def forward(self, input_ids, attention_mask, labels=None):

        # ----------------------------------------------------
        # GPT-2
        #
        # last_hidden_state:
        # [B, T, 768]
        # ----------------------------------------------------

        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        hidden_states = outputs.last_hidden_state

        # ----------------------------------------------------
        # Find the position of the last real token
        #
        # attention_mask:
        #
        #   1 1 1 1 1 0 0
        #
        # last real token = position 4
        # ----------------------------------------------------


        sequence_lengths = attention_mask.sum(dim=1) - 1

        batch_indices = torch.arange(
            hidden_states.size(0),
            device=hidden_states.device,
        )

        # [B, 768]
        pooled_hidden_states = hidden_states[
            batch_indices,
            sequence_lengths,
        ]

        # ----------------------------------------------------
        # Classification head
        #
        # [B, 768] -> [B, 1]
        # ----------------------------------------------------

        logits = self.classifier(
            pooled_hidden_states
        )
        #print(f"in forward => logits: {logits}")
        logits = logits.squeeze(-1)
        #print(f"in forward => logits squeezed: f{logits}")


        # ----------------------------------------------------
        # Classification loss
        # ----------------------------------------------------

        loss = None

        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss()
            loss = loss_fn(logits, labels.float())

        return {
            "loss": loss,
            "logits": logits, #logits dimension squeezed/collpased
        }



model = GPT2SentimentClassifier(
    MODEL_NAME,
)



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [139]:
model

GPT2SentimentClassifier(
  (gpt2): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (classifier): Linear(in_features=768, out_features=1, bias=True)
)

In [140]:
# ============================================================
# Freeze GPT-2
# ============================================================

for parameter in model.gpt2.parameters():
    parameter.requires_grad = False


# Only classifier is trainable.
trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_count = sum(
    p.numel()
    for p in trainable_params
)

print()
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_count)





Total parameters: 124440577
Trainable parameters: 769


In [141]:
# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)



In [142]:
# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate(model, data_loader):

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in data_loader:

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)


        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs["loss"]
        logits = outputs["logits"]
        #print(f"in evaluate=> logits: {logits}")


        predictions = (logits > 0).long() #logits.argmax(dim=-1)
        #print(f"in evaluate=> predictions: {predictions}")
        #print(f"in evaluate=> labels: {labels}")

        total_loss += loss.item() * labels.size(0)

        total_correct += (
            predictions == labels
        ).sum().item()

        total_examples += labels.size(0)

    avg_loss = total_loss / total_examples
    accuracy = total_correct / total_examples


    return avg_loss, accuracy



In [143]:
# ============================================================
# Training
# ============================================================

model.to(DEVICE)

for epoch in range(NUM_EPOCHS):

    model.train()

    running_loss = 0.0

    for step, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)


        # Forward
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs["loss"]

        # Backward
        optimizer.zero_grad()

        loss.backward()

        # Update ONLY the classification head
        optimizer.step()

        running_loss += loss.item()


        if (step + 1) % 100 == 0:

            avg_loss = running_loss / 100

            print(
                f"Epoch {epoch + 1} "
                f"Step {step + 1} "
                f"Loss {avg_loss:.4f}"
            )

            running_loss = 0.0

    val_loss, val_accuracy = evaluate(
        model,
        val_loader,
    )


    print()
    print(
        f"Epoch {epoch + 1} "
        f"Validation Loss: {val_loss:.4f} "
        f"Validation Accuracy: {val_accuracy:.4f}"
    )
    print()


Epoch 1 Step 100 Loss 0.7705
Epoch 1 Step 200 Loss 0.6810
Epoch 1 Step 300 Loss 0.6762
Epoch 1 Step 400 Loss 0.6545
Epoch 1 Step 500 Loss 0.6612
Epoch 1 Step 600 Loss 0.6495
Epoch 1 Step 700 Loss 0.6116
Epoch 1 Step 800 Loss 0.6340
Epoch 1 Step 900 Loss 0.6406
Epoch 1 Step 1000 Loss 0.5989
Epoch 1 Step 1100 Loss 0.6116
Epoch 1 Step 1200 Loss 0.6022
Epoch 1 Step 1300 Loss 0.5982
Epoch 1 Step 1400 Loss 0.6291
Epoch 1 Step 1500 Loss 0.6091
Epoch 1 Step 1600 Loss 0.5978
Epoch 1 Step 1700 Loss 0.5447
Epoch 1 Step 1800 Loss 0.5824
Epoch 1 Step 1900 Loss 0.5756
Epoch 1 Step 2000 Loss 0.5950
Epoch 1 Step 2100 Loss 0.5968
Epoch 1 Step 2200 Loss 0.5847
Epoch 1 Step 2300 Loss 0.5749
Epoch 1 Step 2400 Loss 0.5579
Epoch 1 Step 2500 Loss 0.5617

Epoch 1 Validation Loss: 0.6577 Validation Accuracy: 0.5470



In [144]:
# ============================================================
# Example predictions
# ============================================================

model.eval()

examples = [
    "This movie was absolutely fantastic.",
    "The movie was boring and disappointing.",
    "I really enjoyed this film.",
    "I wouldn't recommend this movie.",
]

encoded = tokenizer(
    examples,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)


encoded = {
    key: value.to(DEVICE)
    for key, value in encoded.items()
}


with torch.no_grad():

    outputs = model(
        input_ids=encoded["input_ids"],
        attention_mask=encoded["attention_mask"],
    )

    logits = outputs["logits"]
    print(f"example predictions=> logits: {logits}")
    probabilities = torch.sigmoid(logits) #.squeeze(-1)  #torch.softmax(
    #    outputs["logits"],
    #    dim=-1,
    #)
    print(f"example predictions=> probabilities: {probabilities}")

    predictions = (probabilities > 0.5).long() #probabilities.argmax(dim=-1)
    print(f"example predictions=> predictions: {predictions}")

for text, prediction, probability in zip(
    examples,
    predictions,
    probabilities
):

    label = (
        "positive"
        if prediction.item() == 1
        else "negative"
    )

    print()
    print("Text:", text)
    print("Prediction:", label)
    print("Probabilities:", probability.cpu().tolist())


example predictions=> logits: tensor([ 2.2236, -0.2233,  2.1784,  0.2719], device='cuda:0')
example predictions=> probabilities: tensor([0.9023, 0.4444, 0.8983, 0.5676], device='cuda:0')
example predictions=> predictions: tensor([1, 0, 1, 1], device='cuda:0')

Text: This movie was absolutely fantastic.
Prediction: positive
Probabilities: 0.9023447036743164

Text: The movie was boring and disappointing.
Prediction: negative
Probabilities: 0.4444046914577484

Text: I really enjoyed this film.
Prediction: positive
Probabilities: 0.8982937335968018

Text: I wouldn't recommend this movie.
Prediction: positive
Probabilities: 0.5675510168075562


In [146]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [147]:
!jupyter nbconvert --to html /content/drive/MyDrive/"Colab Notebooks"/sft_sst2_gpt2_E1-1_1logit.ipynb --output-dir=/content/drive/MyDrive/

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/sft_sst2_gpt2_E1-1_1logit.ipynb to html
[NbConvertApp] Writing 345217 bytes to /content/drive/MyDrive/sft_sst2_gpt2_E1-1_1logit.html
